In [35]:
import time
import warnings
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

X = pd.read_csv("../data/secom.data", sep=" ", header=None)
y = pd.read_csv("../data/secom_labels.data", sep=" ", header=None,
                names=["label", "timestamp"])
y["timestamp"] = pd.to_datetime(y["timestamp"], format="%d/%m/%Y %H:%M:%S")

df = X.copy()
df["label"] = (y["label"] == 1).astype(int)
df["timestamp"] = y["timestamp"]
df = df.sort_values("timestamp").reset_index(drop=True)

split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]

feature_cols = [c for c in df.columns if c not in ("label", "timestamp")]
X_train, y_train = train[feature_cols], train["label"]

keep = [c for c in feature_cols
        if X_train[c].isna().mean() <= 0.5 and X_train[c].nunique() > 1]
X_train = X_train[keep]
print("features:", len(feature_cols), "->", len(keep))

features: 590 -> 444


In [36]:
cv = TimeSeriesSplit(n_splits=5)

def make(model, k=None):
    steps = [("imp", SimpleImputer(strategy="median")),
             ("var", VarianceThreshold(0.0)),    
             ("sc", StandardScaler())]
    if k:
        steps.append(("sel", SelectKBest(f_classif, k=k)))
    steps.append(("clf", model))
    return Pipeline(steps)

def evaluate(name, pipe, cv_obj=cv):
    res = cross_validate(pipe, X_train, y_train, cv=cv_obj,
                         scoring=["roc_auc", "average_precision"])
    print(f"{name:22s} ROC-AUC={res['test_roc_auc'].mean():.3f} (±{res['test_roc_auc'].std():.3f})  "
          f"PR-AUC={res['test_average_precision'].mean():.3f} (±{res['test_average_precision'].std():.3f})")

candidates = {
    "logreg, all features": make(LogisticRegression(class_weight="balanced", C=0.01, max_iter=2000)),
    "logreg, top 20":       make(LogisticRegression(class_weight="balanced", C=0.1, max_iter=2000), k=20),
    "logreg, top 50":       make(LogisticRegression(class_weight="balanced", C=0.1, max_iter=2000), k=50),
    "random forest":        make(RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                                        class_weight="balanced_subsample",
                                                        random_state=42, n_jobs=-1)),
}



In [37]:
for name, pipe in candidates.items():
    evaluate(name, pipe)

logreg, all features   ROC-AUC=0.514 (±0.084)  PR-AUC=0.075 (±0.040)
logreg, top 20         ROC-AUC=0.570 (±0.020)  PR-AUC=0.110 (±0.060)
logreg, top 50         ROC-AUC=0.569 (±0.075)  PR-AUC=0.118 (±0.049)
random forest          ROC-AUC=0.597 (±0.130)  PR-AUC=0.119 (±0.058)


In [38]:
# pre-fold check
for i, (tr, va) in enumerate(cv.split(X_train)):
    print(f"fold {i}: train={len(tr)} (fails {y_train.iloc[tr].sum()}), "
          f"val={len(va)} (fails {y_train.iloc[va].sum()})")

res = cross_validate(candidates["random forest"], X_train, y_train, cv=cv, scoring="roc_auc")
print(res["test_score"].round(3))

fold 0: train=213 (fails 29), val=208 (fails 26)
fold 1: train=421 (fails 55), val=208 (fails 10)
fold 2: train=629 (fails 65), val=208 (fails 6)
fold 3: train=837 (fails 71), val=208 (fails 6)
fold 4: train=1045 (fails 77), val=208 (fails 10)
[0.496 0.465 0.657 0.822 0.544]


In [39]:
# drift check
half = len(X_train) // 2
early, late = X_train.iloc[:half], X_train.iloc[half:]
pvals = []
for c in X_train.columns:
    a, b = early[c].dropna(), late[c].dropna()
    if len(a) > 20 and len(b) > 20:
        pvals.append(ks_2samp(a, b).pvalue)
print(f"{sum(p < 0.001 for p in pvals)} of {len(pvals)} features differ significantly early vs late")

281 of 444 features differ significantly early vs late


In [40]:
more = {
    "logreg L1": Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                           ("clf", LogisticRegression(penalty="l1", solver="liblinear",
                                                      C=0.05, class_weight="balanced"))]),
    "extra trees": make(ExtraTreesClassifier(n_estimators=300, min_samples_leaf=3,
                                             class_weight="balanced_subsample",
                                             random_state=42, n_jobs=-1)),
}
for name, pipe in more.items():
    evaluate(name, pipe)

logreg L1              ROC-AUC=0.551 (±0.110)  PR-AUC=0.090 (±0.047)
extra trees            ROC-AUC=0.592 (±0.076)  PR-AUC=0.100 (±0.038)


In [ ]:
for window in [400, 600, None]:      
    cv_w = TimeSeriesSplit(n_splits=5, max_train_size=window)
    for name in ["logreg, top 50", "random forest"]:
        evaluate(f"w={window} {name}", candidates[name], cv_obj=cv_w)

w=400 logreg, top 50   ROC-AUC=0.566 (±0.078)  PR-AUC=0.107 (±0.064)
w=400 random forest    ROC-AUC=0.548 (±0.054)  PR-AUC=0.085 (±0.046)
w=600 logreg, top 50   ROC-AUC=0.552 (±0.058)  PR-AUC=0.112 (±0.050)
w=600 random forest    ROC-AUC=0.549 (±0.098)  PR-AUC=0.090 (±0.040)
w=None logreg, top 50  ROC-AUC=0.569 (±0.075)  PR-AUC=0.118 (±0.049)
w=None random forest   ROC-AUC=0.597 (±0.130)  PR-AUC=0.119 (±0.058)


In [42]:
block = 200
for start in range(0, len(y_train), block):
    chunk = y_train.iloc[start:start + block]
    print(f"rows {start:4d}-{start + len(chunk):4d}: fails={chunk.sum():2d}  rate={chunk.mean():.3f}")

rows    0- 200: fails=29  rate=0.145
rows  200- 400: fails=25  rate=0.125
rows  400- 600: fails= 9  rate=0.045
rows  600- 800: fails= 6  rate=0.030
rows  800-1000: fails= 7  rate=0.035
rows 1000-1200: fails= 6  rate=0.030
rows 1200-1253: fails= 5  rate=0.094


Findings :
- failure rate fell from about 13% to 4%
- 281 of 444 features drifted
- models tied within noise
- rolling windows didn’t help and gradient boosting wasn’t evaluated